In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import ast
from catboost import CatBoostClassifier

In [14]:
df = pd.read_csv('Cleaned_Data_V2.csv')
if 'validation_status' in df.columns:
    df = pd.get_dummies(df, columns=['validation_status'], prefix='status', dtype=int)
df= df[(df["status_APPROVED"] == 1) | (df["status_REJECTED"] == 1)]
Y = df["status_APPROVED"].values
X = df.drop(columns=['status_APPROVED','status_REJECTED','status_PENDING','status_PROCESSING','status_DUPLICATE','location','data_sent_to_scita_timestamp'])


In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

cat_features = ['vehicle_number','vehicle_type','police_station','data_sent_to_scita','junction_name','updated_vehicle_number','updated_vehicle_type']

model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.3,
    depth=8,
    verbose=10 # Prints log every 10 iterations
)

model.fit(
    X_train, 
    y_train, 
    cat_features=cat_features, 
    eval_set=(X_test, y_test)
)

0:	learn: 0.5736891	test: 0.5768778	best: 0.5768778 (0)	total: 97ms	remaining: 19.3s
10:	learn: 0.4364093	test: 0.4533237	best: 0.4533237 (10)	total: 1.29s	remaining: 22.1s
20:	learn: 0.3941103	test: 0.4226765	best: 0.4226765 (20)	total: 2.38s	remaining: 20.3s
30:	learn: 0.3679424	test: 0.4064692	best: 0.4064692 (30)	total: 3.63s	remaining: 19.8s
40:	learn: 0.3438634	test: 0.3941463	best: 0.3941463 (40)	total: 4.92s	remaining: 19.1s
50:	learn: 0.3166531	test: 0.3800527	best: 0.3800527 (50)	total: 6.16s	remaining: 18s
60:	learn: 0.2936580	test: 0.3665914	best: 0.3665914 (60)	total: 7.56s	remaining: 17.2s
70:	learn: 0.2787603	test: 0.3599541	best: 0.3599541 (70)	total: 8.96s	remaining: 16.3s
80:	learn: 0.2643632	test: 0.3524936	best: 0.3524936 (80)	total: 10.2s	remaining: 15s
90:	learn: 0.2558928	test: 0.3499830	best: 0.3499830 (90)	total: 11.7s	remaining: 14s
100:	learn: 0.2439781	test: 0.3454970	best: 0.3453727 (98)	total: 12.9s	remaining: 12.7s
110:	learn: 0.2314220	test: 0.3418224	be

CatBoostClassifier(depth=8, iterations=200, learning_rate=0.3, verbose=10)

In [16]:
y_pred = model.predict(X_test)
predictions = (y_pred >= 0.5).astype(int)
print("--- Classification Report ---")
print(classification_report(y_test, predictions, target_names=['Rejected', 'Approved']))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, predictions))



--- Classification Report ---
              precision    recall  f1-score   support

    Rejected       0.87      0.80      0.83      2711
    Approved       0.85      0.91      0.88      3523

    accuracy                           0.86      6234
   macro avg       0.86      0.85      0.86      6234
weighted avg       0.86      0.86      0.86      6234

--- Confusion Matrix ---
[[2161  550]
 [ 320 3203]]
